# Aqua Quenchers — Understandable Benchmark Notebook

This notebook checks the **expense classifier**. It answers: **Does it choose useful categories? How long does training take? What still needs testing on the actual tablet?**

Two modes are available:
- **synthetic:** generated classroom examples. You can run this demonstration immediately with the included source. Every result is marked **SYNTHETIC ONLY**; it is not station accuracy or a release model.
- **reviewed:** your approved, anonymized Owner-reviewed JSONL file. Real data stays outside the repository and notebook outputs.

The Python training source is included on `algo-ias`. The Android expense flow is wired to the canonical Java engine with a strict approved-model digest gate. Supplying the reviewed release model and taking actual-tablet measurements remain separate approval/verification tasks. Outputs are intentionally cleared in this shared notebook; a successful run displays actual aggregate summaries locally.

## 1. Open the source and prepare Python

**Colab:** open this notebook, then run the following commands in a separate cell if you have not already obtained the source and installed its dependencies:

```python
!git clone --branch algo-ias --single-branch https://github.com/deo-ganub/Aqua-Quenchers-BMS.git /content/Aqua-Quenchers-BMS
%pip install -r /content/Aqua-Quenchers-BMS/classifier/requirements.txt
```

Use a runtime compatible with the pinned packages. If the repository is private, use your approved GitHub access method; never paste a token into a notebook. Skip cloning if the directory already exists. Restart the runtime if requested after package installation.

**Local:** use the repository's existing Python environment with `classifier/requirements.txt` installed. No cloning or installation happens automatically in the cells below.

The source checks all seven categories: Fuel, Electricity, Water, Maintenance, Supplies, Salary and Others. TF-IDF turns words into numbers; logistic regression learns category weights; weighted Multinomial Naive Bayes is the comparison method.

In [ ]:
from pathlib import Path
import importlib, json, os, subprocess, sys, tempfile, uuid

PROJECT_ROOT = Path("/content/Aqua-Quenchers-BMS") if Path("/content").exists() else Path.cwd()
MODE = "synthetic"          # Change to "reviewed" only for the approved real-data experiment.
DATASET = None               # Reviewed mode: absolute path to private reviewed JSONL.
PRIVATE_WORKSPACE = None     # None creates a new private temporary workspace outside Git.
MODEL_VERSION = "AQ-LR-EXPERIMENT-v1"
OWNER_REVIEW_CONFIRMED = False
FROZEN_REVIEW_CONFIRMED = False

RUN_EXPERIMENT = False       # Set True to generate demo data (or use reviewed data), split and train.
RUN_HELD_OUT_ONCE = False    # Set True only after inspecting the frozen experiment.
RUN_FIVE_PROCESS_TIMING = False  # Optional: one warmup plus five fresh prepare/train processes.
CONTEXT = None

## 2. Private files and repeatable commands

The next cell defines helpers. It does not train or access your real data by itself.

- Source is checked from the actual checkout, including when you start from another directory.
- Each experiment gets a new directory; earlier files and failed-run evidence are preserved.
- Commands use argument lists and a small explicit environment, not shell-built dataset commands or inherited credentials.
- Raw training/evaluation logs remain private. The notebook displays category counts and aggregate metrics, not raw customer remarks or error examples.

For reviewed mode, setting a Boolean is only an acknowledgement; it does not replace the Owner's actual review and privacy decisions. The dataset loader also rejects unreviewed rows, unsupported categories and obvious contact identifiers. That screening alone is not anonymization.

In [ ]:
LABELS = ["Fuel", "Electricity", "Water", "Maintenance", "Supplies", "Salary", "Others"]

def configure():
    if MODE not in ("synthetic", "reviewed"):
        raise ValueError("Choose synthetic or reviewed mode.")
    repo = Path(PROJECT_ROOT).expanduser().resolve(strict=True)
    if not (repo / "classifier/aqua_classifier/pipeline.py").is_file():
        raise ValueError("Open the algo-ias source checkout containing classifier/aqua_classifier first.")
    if not isinstance(MODEL_VERSION, str) or not MODEL_VERSION.strip():
        raise ValueError("Provide a model version for this experiment.")
    source = str(repo / "classifier")
    if source not in sys.path:
        sys.path.insert(0, source)
    module = importlib.import_module("aqua_classifier")
    if Path(module.__file__).resolve().parent != repo / "classifier/aqua_classifier":
        raise ValueError("Another classifier checkout is already imported. Restart the runtime.")
    from aqua_classifier.pipeline import environment
    provenance = environment()
    if MODE == "reviewed" and (not OWNER_REVIEW_CONFIRMED or provenance["source_dirty"]
                                or provenance["source_commit"] == "UNCOMMITTED"):
        raise ValueError("Reviewed mode requires actual Owner review and a clean committed classifier/app source.")
    if PRIVATE_WORKSPACE is None:
        workspace = Path(tempfile.mkdtemp(prefix="aq-private-benchmark-"))
    else:
        workspace = Path(PRIVATE_WORKSPACE).expanduser().resolve()
        if workspace == repo or workspace.is_relative_to(repo):
            raise ValueError("Keep experiment data outside the repository.")
        workspace.mkdir(mode=0o700, parents=False, exist_ok=True)
    if workspace.stat().st_mode & 0o077:
        raise ValueError("Use a workspace restricted to its custodian, with directory permissions 0700.")
    run = workspace / ("run-" + uuid.uuid4().hex)
    run.mkdir(mode=0o700)
    logs = run / "logs"
    logs.mkdir(mode=0o700)
    for name in ("home", "tmp"):
        (run / name).mkdir(mode=0o700)
    if MODE == "synthetic":
        dataset = run / "synthetic-demo.jsonl"
        # Deliberately easy made-up examples: demonstrate code, never claim real accuracy.
        rows = [dict(id=f"demo-{i}-{j}", remarks=f"{label.lower()} station {word}",
                     category=label, reviewed=True, reviewer_id="synthetic-fixture", 
                     group_id=f"demo-{i}-{j}", synthetic=True)
                for i, label in enumerate(LABELS)
                for j, word in enumerate(("amber", "birch", "cedar", "delta", "elm", "fern"))]
        fd = os.open(dataset, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            stream.write("\n".join(json.dumps(row) for row in rows) + "\n")
    else:
        if DATASET is None:
            raise ValueError("Set DATASET to the private reviewed JSONL file.")
        dataset = Path(DATASET).expanduser().resolve(strict=True)
        if dataset.is_relative_to(repo):
            raise ValueError("Reviewed data must be outside the source checkout.")
    if not dataset.is_file() or dataset.stat().st_size > 32 * 1024 * 1024:
        raise ValueError("Dataset must be an ordinary JSONL file at most 32MiB.")
    return dict(repo=repo, run=run, logs=logs, dataset=dataset, experiment=run / "experiment",
                timing=run / "training-timing.json", synthetic=MODE == "synthetic", provenance=provenance)

def run_command(context, arguments, log_name, timing=False):
    env = {"PATH": str(Path(sys.executable).parent) + ":/usr/bin:/bin",
           "HOME": str(context["run"] / "home"), "TMPDIR": str(context["run"] / "tmp"),
           "PYTHONPATH": str(context["repo"] / "classifier"), "PYTHONNOUSERSITE": "1",
           "PYTHONDONTWRITEBYTECODE": "1", "LANG": "C.UTF-8",
           "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}
    module = "aqua_classifier.benchmark" if timing else "aqua_classifier"
    argv = [sys.executable, "-B", "-m", module, *arguments]
    if context["synthetic"]:
        argv.append("--synthetic")
    fd = os.open(context["logs"] / log_name, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as stream:
        completed = subprocess.run(argv, cwd=context["repo"], env=env, stdout=stream,
                                   stderr=subprocess.STDOUT, timeout=900, check=False)
    if completed.returncode:
        raise RuntimeError("The command failed. Review its private log and any freeze/evaluation marker before retrying.")

def report_json(path):
    if path.stat().st_size > 8 * 1024 * 1024:
        raise ValueError("Report exceeds this notebook's display-reader limit.")
    return json.loads(path.read_text(encoding="utf-8"))

def evidence_label(context):
    return "SYNTHETIC ONLY — classroom demonstration" if context["synthetic"] else "OWNER-REVIEWED EXPERIMENT — release approval separate"

## 3. Prepare the examples and train both methods

Set `RUN_EXPERIMENT = True` in the settings cell, then run the helper and training cells.

The grouped split targets **60% training / 20% validation / 20% testing**, with seed42. Exact and near duplicates, receipt families and declared groups stay together. Actual counts may differ because groups must stay together. Vocabulary and IDF are learned from training data only. Model and suggestion settings are selected using validation data, never the held-out test.

Synthetic mode creates **42 made-up rows** for this demonstration. A row's `reviewed` flag is part of the fixture schema, not evidence of an Owner reviewing real records. Reviewed mode generates no replacement examples and uses only the supplied file.

In [ ]:
if RUN_EXPERIMENT:
    CONTEXT = configure()
    args = ["--dataset", str(CONTEXT["dataset"]), "--output", str(CONTEXT["experiment"])]
    run_command(CONTEXT, ["prepare", *args], "01-prepare.log")
    run_command(CONTEXT, ["train", *args, "--model-version", MODEL_VERSION], "02-train.log")
    print(evidence_label(CONTEXT))
    print("Preparation and training completed. Private output directory:", CONTEXT["run"])
else:
    print("Ready: enable RUN_EXPERIMENT for a synthetic demonstration or configured reviewed experiment.")

## 4. Understand the training output

This summary shows the number of examples in each split and category, the vocabulary size, and model size. Validation metrics describe the examples used to choose settings; they are **not final held-out results**.

**Macro-F1** gives each category equal importance. **Recall** asks how many examples belonging to a category were found. **Suggestion coverage** tells how often the frozen policy offers a suggestion rather than leaving a manual choice. The printed values below come from actual generated report files.

In [ ]:
if CONTEXT is not None:
    from aqua_classifier.data import load_dataset
    rows = load_dataset(CONTEXT["dataset"], CONTEXT["synthetic"])
    split = report_json(CONTEXT["experiment"] / "split.json")
    training = report_json(CONTEXT["experiment"] / "training.json")
    by_id = {row["id"]: row["category"] for row in rows}
    support = {part: {label: sum(by_id[item] == label for item in ids) for label in LABELS}
               for part, ids in split["partitions"].items()}
    TRAINING_SUMMARY = {"evidence": evidence_label(CONTEXT), "split_counts": {k: len(v) for k, v in split["partitions"].items()},
                        "category_support": support, "vocabulary_size": training["vocabulary_size"],
                        "model_bytes": training["model_bytes"], "source_commit": training["environment"]["source_commit"],
                        "source_dirty": training["environment"]["source_dirty"],
                        "validation_only": {"logistic_macro_f1": training["primary_validation"]["macro_f1"],
                                            "naive_bayes_macro_f1": training["comparison_validation"]["macro_f1"]},
                        "release_approved": False}
    print(json.dumps(TRAINING_SUMMARY, indent=2))
else:
    print("The training summary appears after an actual training run.")

## 5. Run the frozen held-out evaluation once

Inspect the frozen split, model, policy and gates first. Set `RUN_HELD_OUT_ONCE = True` to evaluate the reserved examples. Reviewed mode also requires `FROZEN_REVIEW_CONFIRMED = True` after the actual review.

The CLI reserves the evaluation before reading the test labels. A repeat or interrupted run requires review; do not delete markers or change thresholds to obtain a better result. This cell does not retrain. Synthetic evaluation is a code demonstration only, even if it scores highly on the easy fixtures.

In [ ]:
if RUN_HELD_OUT_ONCE:
    if CONTEXT is None:
        raise ValueError("Prepare and freeze this experiment first.")
    if not CONTEXT["synthetic"] and not FROZEN_REVIEW_CONFIRMED:
        raise ValueError("Review the frozen real-data experiment before using the held-out set.")
    run_command(CONTEXT, ["evaluate", "--dataset", str(CONTEXT["dataset"]),
                         "--output", str(CONTEXT["experiment"])], "03-held-out.log")
    print(evidence_label(CONTEXT), "— held-out evaluation completed once.")
else:
    print("Held-out evaluation stays reserved until you enable it after the freeze review.")

## 6. Read the final aggregate results

The tables compare logistic regression with Naive Bayes using the same test split. A confusion matrix shows correct categories against predicted categories; its row/column order is the seven labels printed below. No raw remarks, example IDs or private error-analysis records are displayed.

The proposed gates remain **macro-F1 >=0.80** and **each category recall >=0.60**. Keep the original gates if the model fails. A synthetic pass never establishes station readiness.

In [ ]:
if CONTEXT is not None and (CONTEXT["experiment"] / "test-results.json").is_file():
    result = report_json(CONTEXT["experiment"] / "test-results.json")
    QUALITY_SUMMARY = {"evidence": evidence_label(CONTEXT), "test_examples": result["test_examples"],
                       "labels": LABELS, "models": {}, "device_benchmark": "NOT RUN", "release_approved": False}
    for key in ("primary", "comparison"):
        metrics = result[key]
        QUALITY_SUMMARY["models"][key] = {name: metrics[name] for name in
            ("accuracy", "macro_f1", "per_class", "confusion_matrix", "suggestion_coverage", "error_among_offered", "quality_gate_pass")}
    print(json.dumps(QUALITY_SUMMARY, indent=2))
else:
    print("Final test summaries appear only after the held-out procedure completes.")

## 7. Measure training time separately

Set `RUN_FIVE_PROCESS_TIMING = True` to run **one warm-up plus five measured fresh processes**. Each process prepares and trains/tunes on the dataset. This measures the whole preparation-and-training workflow, not just a single optimizer step. The output reports actual durations, median/range and peak resident memory. This does not evaluate the held-out set, and repeated timing runs are not more accuracy examples.

In [ ]:
if RUN_FIVE_PROCESS_TIMING:
    if CONTEXT is None:
        raise ValueError("Configure the experiment first.")
    run_command(CONTEXT, ["--dataset", str(CONTEXT["dataset"]), "--output", str(CONTEXT["timing"])],
                "04-training-timing.log", timing=True)
    timing = report_json(CONTEXT["timing"])
    TIMING_SUMMARY = {"evidence": evidence_label(CONTEXT), "scope": "preparation + training + tuning",
                      **{key: timing[key] for key in ("warmup_processes", "fresh_timed_processes", "median_seconds", "range_seconds", "peak_rss_bytes", "runs", "environment")},
                      "device_benchmark": "NOT RUN"}
    print(json.dumps(TIMING_SUMMARY, indent=2))
else:
    print("Training timing is optional and separate from held-out quality.")

## 8. What still belongs on the actual tablet?

Python/Colab timing is a computer measurement. It cannot establish Android tablet speed or memory use. On the declared Owner tablet with an approved model, run100warm-ups and1000timed inferences; report median/p95 and incremental Java heap, with separate Android process RSS/PSS. Heap is not process memory.

| Acceptance item | Evidence required |
|---|---|
| Macro-F1 >=0.80; every recall >=0.60 | Actual reviewed held-out corpus and frozen evaluation |
| Approved model <=2MiB | Actual release artifact; loader/export cap already implemented |
| Owner-tablet p95 <=100ms | Actual tablet/model measurements |
| Incremental memory <=32MiB | Actual tablet heap and separate process-memory measurements |
| Python/Java/Android agreement | Frozen real model and target runtime parity |

The notebook preserves manual expense entry and does not save expenses, change stock, upload customer records or approve release models. Keep the output directories private and remove private notebook outputs before sharing. Terms: *classifier* chooses categories; *inference* uses a trained model; *macro-F1* summarizes category quality; *p95* is a response-time percentile.